# Inference

Analysis estimating the causal effect of natural disasters on county-level mortgage delinquency rates.

All models use **two-way fixed-effects Panel OLS** (county + month fixed effects) with standard errors clustered at the county level.

In [1]:
%reload_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

from climatefinance import utils

results = []

In [2]:
# Read data
analysis_df = utils.load_analysis_data()
# Fix datetime loading from csv file
analysis_df["month"] = pd.to_datetime(analysis_df["month"])

Loaded 73899 rows from data/analysis/finance_disaster_analysis.csv


In [3]:
panel_df = analysis_df.copy()
panel_df = panel_df.set_index(["fips", "month"]).sort_index()

## Model 1 — Baseline Event Occurrence

Test whether disaster occurrence alone has a detectable average effect on delinquency.

In [4]:
y1 = panel_df["Early_Delinquency_Rate"]
X1 = panel_df[["event_occur"]]

# Early delinquency ~ event_occur

mod1 = PanelOLS(
    y1,
    X1,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res1 = mod1.fit(cov_type="clustered", cluster_entity=True)
print(res1.summary)
results.extend(utils.extract_panel_results(res1, "baseline", "Early_Delinquency_Rate"))

                            PanelOLS Estimation Summary                             
Dep. Variable:     Early_Delinquency_Rate   R-squared:                     6.368e-05
Estimator:                       PanelOLS   R-squared (Between):             -0.0004
No. Observations:                   73899   R-squared (Within):           -7.394e-05
Date:                    Sat, Mar 21 2026   R-squared (Overall):             -0.0003
Time:                            11:00:50   Log-likelihood                -4.694e+04
Cov. Estimator:                 Clustered                                           
                                            F-statistic:                      4.6700
Entities:                             357   P-value                           0.0307
Avg Obs:                           207.00   Distribution:                 F(1,73335)
Min Obs:                           207.00                                           
Max Obs:                           207.00   F-statistic (robust):

In [5]:
y2 = panel_df["Late_Delinquency_Rate"]
X2 = panel_df[["event_occur"]]

# Late delinquency ~ event_occur

mod2 = PanelOLS(
    y2,
    X2,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res2 = mod2.fit(cov_type="clustered", cluster_entity=True)
print(res2.summary)
results.extend(utils.extract_panel_results(res2, "baseline", "Late_Delinquency_Rate"))

                            PanelOLS Estimation Summary                            
Dep. Variable:     Late_Delinquency_Rate   R-squared:                     2.011e-06
Estimator:                      PanelOLS   R-squared (Between):             -0.0001
No. Observations:                  73899   R-squared (Within):           -1.544e-05
Date:                   Sat, Mar 21 2026   R-squared (Overall):          -9.415e-05
Time:                           11:00:50   Log-likelihood                -8.657e+04
Cov. Estimator:                Clustered                                           
                                           F-statistic:                      0.1475
Entities:                            357   P-value                           0.7010
Avg Obs:                          207.00   Distribution:                 F(1,73335)
Min Obs:                          207.00                                           
Max Obs:                          207.00   F-statistic (robust):            

## Model 2 — Distributed Lag (Damage Severity)

The binary indicator may be too coarse — disasters vary widely in severity and their effects may be distributed over time.
Try with 0–6 month lags of `log_total_damage` and estimate the dynamic response of delinquency to disaster damage.

In [6]:
panel_df = analysis_df.copy()
panel_df = panel_df.set_index(["fips", "month"]).sort_index()

# 0 to 6 month lags of log damage
for k in range(1, 7):
    panel_df[f"log_total_damage_lag{k}"] = (
        panel_df.groupby(level=0)["log_total_damage"].shift(k).fillna(0)
    )

lag_cols = ["log_total_damage"] + [f"log_total_damage_lag{k}" for k in range(1, 7)]

mod_dl = PanelOLS(
    panel_df["Early_Delinquency_Rate"],
    panel_df[lag_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_dl = mod_dl.fit(cov_type="clustered", cluster_entity=True)
print(res_dl.summary)
results.extend(utils.extract_panel_results(res_dl, "distributed_lag", "Early_Delinquency_Rate"))

                            PanelOLS Estimation Summary                             
Dep. Variable:     Early_Delinquency_Rate   R-squared:                        0.0018
Estimator:                       PanelOLS   R-squared (Between):             -0.0052
No. Observations:                   73899   R-squared (Within):              -0.0012
Date:                    Sat, Mar 21 2026   R-squared (Overall):             -0.0046
Time:                            11:00:51   Log-likelihood                -4.688e+04
Cov. Estimator:                 Clustered                                           
                                            F-statistic:                      18.701
Entities:                             357   P-value                           0.0000
Avg Obs:                           207.00   Distribution:                 F(7,73329)
Min Obs:                           207.00                                           
Max Obs:                           207.00   F-statistic (robust):

### Model 2 — Findings (Early Delinquency)

The 1–4 month lags are negative and statistically significant:

- lag 1: -0.0030
- lag 2: -0.0036
- lag 3: -0.0045
- lag 4: -0.0033

while:
- lags 5–6 are not significant

So larger disaster damage is followed by a temporary decline in early delinquency, peaking around 2–4 months later. This pattern is consistent with post-disaster mortgage forbearance programs and FEMA aid temporarily easing borrowers' financial burden.

### Distributed Lag — Late Delinquency

Same lag structure applied to `Late_Delinquency_Rate` to check whether the damage effect extends to late delinquency.

In [7]:
panel_df = analysis_df.copy()
panel_df = panel_df.set_index(["fips", "month"]).sort_index()

# 0 to 6 month lags of log damage
for k in range(1, 7):
    panel_df[f"log_total_damage_lag{k}"] = (
        panel_df.groupby(level=0)["log_total_damage"].shift(k).fillna(0)
    )

mod_l_late = PanelOLS(
    panel_df["Late_Delinquency_Rate"],
    panel_df[lag_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_l_late = mod_l_late.fit(cov_type="clustered", cluster_entity=True)
print(res_l_late.summary)
results.extend(utils.extract_panel_results(res_l_late, "distributed_lag", "Late_Delinquency_Rate"))

                            PanelOLS Estimation Summary                            
Dep. Variable:     Late_Delinquency_Rate   R-squared:                     4.602e-05
Estimator:                      PanelOLS   R-squared (Between):             -0.0013
No. Observations:                  73899   R-squared (Within):              -0.0001
Date:                   Sat, Mar 21 2026   R-squared (Overall):             -0.0009
Time:                           11:00:52   Log-likelihood                -8.657e+04
Cov. Estimator:                Clustered                                           
                                           F-statistic:                      0.4821
Entities:                            357   P-value                           0.8483
Avg Obs:                          207.00   Distribution:                 F(7,73329)
Min Obs:                          207.00                                           
Max Obs:                          207.00   F-statistic (robust):            

## Model 3 — Disaster-Type Effects

Different disaster types may affect delinquency differently.
We create binary indicators for flood, tornado, thunderstorm, and hail occurrence, then estimate their effects individually and jointly.

### Individual type regressions

Each disaster type is regressed separately against both early and late delinquency to identify which types drive effects on their own.

In [8]:

panel_df["flood_occur"] = (panel_df["n_flood"] > 0).astype(int)
panel_df["tornado_occur"] = (panel_df["n_tornado"] > 0).astype(int)
panel_df["thunder_occur"] = (panel_df["n_thunderstorm"] > 0).astype(int)
panel_df["hail_occur"] = (panel_df["n_hail"] > 0).astype(int)

X_types = panel_df[[ "flood_occur"]]

mod_types = PanelOLS(
    panel_df["Early_Delinquency_Rate"],
    X_types,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_types = mod_types.fit(cov_type="clustered", cluster_entity=True)
print(res_types.summary)
results.extend(utils.extract_panel_results(res_types, "flood_only", "Early_Delinquency_Rate"))

                            PanelOLS Estimation Summary                             
Dep. Variable:     Early_Delinquency_Rate   R-squared:                     7.593e-05
Estimator:                       PanelOLS   R-squared (Between):             -0.0003
No. Observations:                   73899   R-squared (Within):           -8.872e-05
Date:                    Sat, Mar 21 2026   R-squared (Overall):             -0.0002
Time:                            11:00:53   Log-likelihood                -4.694e+04
Cov. Estimator:                 Clustered                                           
                                            F-statistic:                      5.5687
Entities:                             357   P-value                           0.0183
Avg Obs:                           207.00   Distribution:                 F(1,73335)
Min Obs:                           207.00                                           
Max Obs:                           207.00   F-statistic (robust):

In [9]:
X_types = panel_df[[ "flood_occur"]]

mod_types = PanelOLS(
    panel_df["Late_Delinquency_Rate"],
    X_types,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_types = mod_types.fit(cov_type="clustered", cluster_entity=True)
print(res_types.summary)
results.extend(utils.extract_panel_results(res_types, "flood_only", "Late_Delinquency_Rate"))

                            PanelOLS Estimation Summary                            
Dep. Variable:     Late_Delinquency_Rate   R-squared:                     3.617e-07
Estimator:                      PanelOLS   R-squared (Between):          -4.027e-05
No. Observations:                  73899   R-squared (Within):           -7.901e-06
Date:                   Sat, Mar 21 2026   R-squared (Overall):          -2.859e-05
Time:                           11:00:53   Log-likelihood                -8.657e+04
Cov. Estimator:                Clustered                                           
                                           F-statistic:                      0.0265
Entities:                            357   P-value                           0.8706
Avg Obs:                          207.00   Distribution:                 F(1,73335)
Min Obs:                          207.00                                           
Max Obs:                          207.00   F-statistic (robust):            

In [10]:
# Hail
X_types = panel_df[[ "hail_occur"]]

mod_types = PanelOLS(
    panel_df["Early_Delinquency_Rate"],
    X_types,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_types = mod_types.fit(cov_type="clustered", cluster_entity=True)
print(res_types.summary)
results.extend(utils.extract_panel_results(res_types, "hail_only", "Early_Delinquency_Rate"))

X_types = panel_df[[ "hail_occur"]]

mod_types = PanelOLS(
    panel_df["Late_Delinquency_Rate"],
    X_types,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_types = mod_types.fit(cov_type="clustered", cluster_entity=True)
print(res_types.summary)
results.extend(utils.extract_panel_results(res_types, "hail_only", "Late_Delinquency_Rate"))

                            PanelOLS Estimation Summary                             
Dep. Variable:     Early_Delinquency_Rate   R-squared:                     9.325e-05
Estimator:                       PanelOLS   R-squared (Between):              0.0002
No. Observations:                   73899   R-squared (Within):               0.0001
Date:                    Sat, Mar 21 2026   R-squared (Overall):              0.0001
Time:                            11:00:54   Log-likelihood                -4.694e+04
Cov. Estimator:                 Clustered                                           
                                            F-statistic:                      6.8392
Entities:                             357   P-value                           0.0089
Avg Obs:                           207.00   Distribution:                 F(1,73335)
Min Obs:                           207.00                                           
Max Obs:                           207.00   F-statistic (robust):

In [11]:
# Tornado
X_types = panel_df[[ "tornado_occur"]]

mod_types = PanelOLS(
    panel_df["Early_Delinquency_Rate"],
    X_types,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_types = mod_types.fit(cov_type="clustered", cluster_entity=True)
print(res_types.summary)
results.extend(utils.extract_panel_results(res_types, "tornado_only", "Early_Delinquency_Rate"))

X_types = panel_df[[ "tornado_occur"]]

mod_types = PanelOLS(
    panel_df["Late_Delinquency_Rate"],
    X_types,
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
res_types = mod_types.fit(cov_type="clustered", cluster_entity=True)
print(res_types.summary)
results.extend(utils.extract_panel_results(res_types, "tornado_only", "Late_Delinquency_Rate"))

                            PanelOLS Estimation Summary                             
Dep. Variable:     Early_Delinquency_Rate   R-squared:                     2.759e-07
Estimator:                       PanelOLS   R-squared (Between):          -1.303e-05
No. Observations:                   73899   R-squared (Within):           -2.912e-06
Date:                    Sat, Mar 21 2026   R-squared (Overall):           -1.15e-05
Time:                            11:00:55   Log-likelihood                -4.694e+04
Cov. Estimator:                 Clustered                                           
                                            F-statistic:                      0.0202
Entities:                             357   P-value                           0.8869
Avg Obs:                           207.00   Distribution:                 F(1,73335)
Min Obs:                           207.00                                           
Max Obs:                           207.00   F-statistic (robust):

### Joint type model

All four disaster types entered simultaneously. This reveals which types retain significance when competing for explanatory power.

In [12]:
type_cols = ["flood_occur", "tornado_occur", "thunder_occur", "hail_occur"]

mod_types_early = PanelOLS(
    panel_df["Early_Delinquency_Rate"],
    panel_df[type_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)

res_types_early = mod_types_early.fit(cov_type="clustered", cluster_entity=True)
print(res_types_early.summary)
results.extend(
    utils.extract_panel_results(res_types_early, "joint_types", "Early_Delinquency_Rate")
)

mod_types_late = PanelOLS(
    panel_df["Late_Delinquency_Rate"],
    panel_df[type_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)

res_types_late = mod_types_late.fit(cov_type="clustered", cluster_entity=True)
print(res_types_late.summary)
results.extend(
    utils.extract_panel_results(res_types_late, "joint_types", "Late_Delinquency_Rate")
)

                            PanelOLS Estimation Summary                             
Dep. Variable:     Early_Delinquency_Rate   R-squared:                        0.0002
Estimator:                       PanelOLS   R-squared (Between):             -0.0003
No. Observations:                   73899   R-squared (Within):            5.189e-05
Date:                    Sat, Mar 21 2026   R-squared (Overall):             -0.0002
Time:                            11:00:57   Log-likelihood                -4.693e+04
Cov. Estimator:                 Clustered                                           
                                            F-statistic:                      3.9209
Entities:                             357   P-value                           0.0035
Avg Obs:                           207.00   Distribution:                 F(4,73332)
Min Obs:                           207.00                                           
Max Obs:                           207.00   F-statistic (robust):

### Model 3 — Findings

In joint type-specific FE models, flood exposure is associated with a statistically significant decline in early delinquency, while no comparable effect appears for late delinquency. Tornado and hail do not show robust average effects, and thunderstorm exposure is only marginally associated with early delinquency. This suggests flood is the primary disaster channel affecting mortgage performance.

## Model 4 — County Heterogeneity (Flood Interactions)

The average flood effect may mask substantial variation across counties. Identify the top 5 most disaster-exposed counties (by cumulative damage) and interact `flood_occur` with county indicators to estimate county-specific flood effects.

In [13]:
df = analysis_df.copy()
df["month"] = pd.to_datetime(df["month"])

# flood indicator
df["flood_occur"] = (df["n_flood"] > 0).astype(int)

county_exposure = (
    df.groupby(["fips", "County"], as_index=False)
      .agg(
          total_damage_sum=("total_damage", "sum"),
          treated_months=("event_occur", "sum"),
          total_disaster_events=("n_disasters", "sum")
      )
)

top5 = county_exposure.sort_values("total_damage_sum", ascending=False).head(5)
print(top5)

top5_fips = top5["fips"].tolist()

      fips             County  total_damage_sum  treated_months  \
308  48157   Fort Bend County      8.008842e+09             6.0   
314  48339  Montgomery County      7.202165e+09             6.0   
311  48245   Jefferson County      5.278500e+09             9.0   
190  34013       Essex County      5.007000e+09             3.0   
294  47037    Davidson County      2.736081e+09            13.0   

     total_disaster_events  
308                   11.0  
314                    6.0  
311                   11.0  
190                    3.0  
294                   17.0  


In [14]:
df_int = df.copy()

for f in top5_fips:
    df_int[f"flood_x_{f}"] = ((df_int["fips"] == f).astype(int) * df_int["flood_occur"])

panel_int = df_int.set_index(["fips", "month"]).sort_index()

interaction_cols = ["flood_occur"] + [f"flood_x_{f}" for f in top5_fips]

### Interaction model estimation

Regress `Early_Delinquency_Rate` on `flood_occur` plus interaction terms `flood_occur × county_i` for each top-5 county. The base `flood_occur` coefficient captures the average flood effect for all other counties; each interaction term shows how a specific county deviates from that average.

In [15]:
mod_county_het = PanelOLS(
    panel_int["Early_Delinquency_Rate"],
    panel_int[interaction_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)

res_county_het = mod_county_het.fit(cov_type="clustered", cluster_entity=True)
print(res_county_het.summary)
results.extend(
    utils.extract_panel_results(res_county_het, "county_heterogeneity", "Early_Delinquency_Rate")
)

                            PanelOLS Estimation Summary                             
Dep. Variable:     Early_Delinquency_Rate   R-squared:                        0.0001
Estimator:                       PanelOLS   R-squared (Between):             -0.0003
No. Observations:                   73899   R-squared (Within):            -4.17e-05
Date:                    Sat, Mar 21 2026   R-squared (Overall):             -0.0002
Time:                            11:00:58   Log-likelihood                -4.694e+04
Cov. Estimator:                 Clustered                                           
                                            F-statistic:                      1.6188
Entities:                             357   P-value                           0.1373
Avg Obs:                           207.00   Distribution:                 F(6,73330)
Min Obs:                           207.00                                           
Max Obs:                           207.00   F-statistic (robust):

### County-specific total flood effects

Compute the total flood effect for each top-5 county (base coefficient + interaction term) to see whether the flood–delinquency relationship is uniformly negative or varies in sign across high-exposure counties.

In [16]:
base_effect = res_county_het.params["flood_occur"]

rows = []
for f in top5_fips:
    interaction_name = f"flood_x_{f}"
    county_name = top5.loc[top5["fips"] == f, "County"].iloc[0]
    county_effect = base_effect + res_county_het.params.get(interaction_name, 0.0)

    rows.append({
        "fips": f,
        "County": county_name,
        "base_flood_effect": base_effect,
        "interaction": res_county_het.params.get(interaction_name, np.nan),
        "county_flood_effect": county_effect
    })

county_effects_table = pd.DataFrame(rows)
print(county_effects_table)

    fips             County  base_flood_effect  interaction  \
0  48157   Fort Bend County           -0.04628     0.097078   
1  48339  Montgomery County           -0.04628     0.071450   
2  48245   Jefferson County           -0.04628    -0.425200   
3  34013       Essex County           -0.04628     0.212193   
4  47037    Davidson County           -0.04628     0.040624   

   county_flood_effect  
0             0.050799  
1             0.025170  
2            -0.471479  
3             0.165913  
4            -0.005656  


### Model 4 — Findings

This suggests that the effect of flood is not homogeneous across counties.

- On average, flood is associated with a decline in `Early_Delinquency_Rate`, but that relationship varies substantially across counties.
- Some counties show a more negative effect than the average, while others even show a positive effect.
- County-level characteristics such as local income levels, lender composition, disaster preparedness, and relief access likely moderate the aggregate relationship.

In [17]:
# Save all inference results
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
utils.save_analysis_inference(results_df, index=False)

               model                outcome             regressor      coef       se       pvalue  nobs           r2
            baseline Early_Delinquency_Rate           event_occur -0.029563 0.015515 5.672615e-02 73899 6.367621e-05
            baseline  Late_Delinquency_Rate           event_occur -0.008982 0.027202 7.412584e-01 73899 2.010930e-06
     distributed_lag Early_Delinquency_Rate      log_total_damage -0.002161 0.001056 4.072478e-02 73899 1.782029e-03
     distributed_lag Early_Delinquency_Rate log_total_damage_lag1 -0.004052 0.001121 3.018380e-04 73899 1.782029e-03
     distributed_lag Early_Delinquency_Rate log_total_damage_lag2 -0.004977 0.001267 8.546299e-05 73899 1.782029e-03
     distributed_lag Early_Delinquency_Rate log_total_damage_lag3 -0.005848 0.001242 2.477138e-06 73899 1.782029e-03
     distributed_lag Early_Delinquency_Rate log_total_damage_lag4 -0.004540 0.001224 2.083631e-04 73899 1.782029e-03
     distributed_lag Early_Delinquency_Rate log_total_damage_lag